# IA_agente_Camanchaca6.ipynb
## IL3.2 - Análisis de Trazabilidad y Logs
### Proyecto: Sistema Agente Camanchaca - Monitoreo Climático

Este notebook implementa **trazabilidad** para el agente Camanchaca: generación de trace IDs únicos, registro estructurado en JSON de cada etapa del procesamiento, y un analizador de trazas que identifica puntos de falla y oportunidades de mejora, según el indicador IE10 de la EFT (IL3.2).

**Conceptos clave aplicados:**
- Trace IDs únicos por consulta
- Línea temporal de ejecución por etapas
- Correlación de eventos (validación → razonamiento → herramienta → respuesta)
- Analizador de trazas para auditoría


In [1]:
!pip install openai langchain langchain-openai langgraph requests python-dotenv -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\lenov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# SECCIÓN 1: CONFIGURACIÓN BASE
# ============================================================

import os
import uuid
import json
import time
import requests
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

if not os.getenv("OPENAI_BASE_URL"):
    raise ValueError("Falta OPENAI_BASE_URL en .env")
if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("Falta GITHUB_TOKEN en .env")

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado.")
print(f"Modelo: {llm.model_name}")


✓ Modelo configurado.
Modelo: gpt-4o


In [3]:
# ============================================================
# SECCIÓN 2: HERRAMIENTAS DEL AGENTE CAMANCHACA
# (Reutilizadas de los notebooks anteriores)
# ============================================================

CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}


@tool
def get_clima_actual(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    try:
        response = requests.get(url, timeout=10)
        data    = response.json()
        current = data["current"]
        temp      = current["temperature_2m"]
        viento    = current["wind_speed_10m"]
        lluvia    = current["precipitation"]
        codigo    = current["weathercode"]
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


@tool
def evaluar_operacion(centro: str, operacion: str) -> str:
    """Evalúa si las condiciones climáticas son seguras para realizar una operación.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    try:
        response = requests.get(url, timeout=10)
        data    = response.json()
        current = data["current"]
        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        temp   = current["temperature_2m"]
        alertas = []
        if viento > 40:
            alertas.append(f"Viento peligroso: {viento} km/h")
        if lluvia > 10:
            alertas.append(f"Lluvia intensa: {lluvia} mm")
        if temp < 5:
            alertas.append(f"Temperatura muy baja: {temp}°C")
        if temp > 18:
            alertas.append(f"Temperatura elevada: {temp}°C")
        if not alertas:
            return f"Condiciones APTAS para {operacion} en {datos['nombre']}."
        return f"Condiciones NO APTAS para {operacion} en {datos['nombre']}: " + "; ".join(alertas)
    except Exception as e:
        return f"Error al evaluar condiciones: {e}"


tools = [get_clima_actual, evaluar_operacion]
agent_executor = create_react_agent(llm, tools)

print("✓ Herramientas y agente listos.")
print(f"  Herramientas: {[t.name for t in tools]}")


✓ Herramientas y agente listos.
  Herramientas: ['get_clima_actual', 'evaluar_operacion']


C:\Users\lenov\AppData\Local\Temp\ipykernel_23608\2114491952.py:86: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [4]:
# ============================================================
# SECCIÓN 3: MODELO DE TRAZA
# Ref: 1-traceability_analysis.py del repositorio de la materia
# ============================================================

@dataclass
class Evento:
    """Evento individual dentro de una traza."""
    etapa:       str
    inicio_ms:   float
    duracion_ms: float
    estado:      str   # "ok" o "error"
    detalle:     str = ""


@dataclass
class Traza:
    """Traza completa de una petición al agente Camanchaca."""
    trace_id:        str
    timestamp:       str
    mensaje_entrada: str
    eventos:         List[Evento] = field(default_factory=list)
    respuesta_final: Optional[str] = None

    def agregar_evento(self, etapa: str, inicio_ms: float,
                        duracion_ms: float, estado: str = "ok", detalle: str = ""):
        self.eventos.append(Evento(etapa, round(inicio_ms, 2),
                                    round(duracion_ms, 2), estado, detalle))

    def duracion_total_ms(self) -> float:
        return round(sum(e.duracion_ms for e in self.eventos), 2)

    def a_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)


print("✓ Modelo de Traza y Evento definido.")


✓ Modelo de Traza y Evento definido.


In [5]:
# ============================================================
# SECCIÓN 4: AGENTE TRAZABLE CAMANCHACA
# Genera trazas estructuradas para cada consulta real del agente
# ============================================================

class AgenteTrazableCamanchaca:
    """Agente Camanchaca que genera trazas estructuradas para cada petición."""

    def __init__(self, agent_executor):
        self.agent = agent_executor
        self.historial_trazas: List[Traza] = []

    def procesar(self, mensaje: str) -> Traza:
        """Procesa un mensaje del operador registrando trazabilidad por etapas."""
        trace_id = str(uuid.uuid4())[:12]
        traza = Traza(
            trace_id=trace_id,
            timestamp=datetime.now().isoformat(),
            mensaje_entrada=mensaje,
        )

        tiempo_base = time.perf_counter()

        # Etapa 1: Validación de entrada
        inicio = (time.perf_counter() - tiempo_base) * 1000
        es_valida = len(mensaje.strip()) > 0
        duracion = (time.perf_counter() - tiempo_base) * 1000 - inicio
        traza.agregar_evento("validacion_entrada", inicio, duracion,
                              "ok" if es_valida else "error",
                              f"Longitud: {len(mensaje)} caracteres")

        if not es_valida:
            traza.respuesta_final = "[ERROR] Mensaje vacío"
            self.historial_trazas.append(traza)
            return traza

        # Etapa 2: Identificación de centro mencionado
        inicio = (time.perf_counter() - tiempo_base) * 1000
        centro_detectado = None
        for centro in CENTROS:
            if centro in mensaje.lower():
                centro_detectado = centro
                break
        duracion = (time.perf_counter() - tiempo_base) * 1000 - inicio
        traza.agregar_evento("identificacion_centro", inicio, duracion, "ok",
                              f"Centro detectado: {centro_detectado or 'ninguno explícito'}")

        # Etapa 3: Invocación del agente (LLM + herramientas)
        inicio = (time.perf_counter() - tiempo_base) * 1000
        try:
            response = self.agent.invoke({"messages": [{"role": "user", "content": mensaje}]})
            output   = response["messages"][-1].content
            duracion = (time.perf_counter() - tiempo_base) * 1000 - inicio
            traza.agregar_evento("invocacion_agente", inicio, duracion, "ok",
                                  f"Modelo: gpt-4o, longitud_respuesta: {len(output)}")
            traza.respuesta_final = output
        except Exception as e:
            duracion = (time.perf_counter() - tiempo_base) * 1000 - inicio
            traza.agregar_evento("invocacion_agente", inicio, duracion, "error", str(e))
            traza.respuesta_final = f"[ERROR] {e}"

        self.historial_trazas.append(traza)
        return traza


agente_trazable = AgenteTrazableCamanchaca(agent_executor)
print("✓ Agente trazable Camanchaca listo.")


✓ Agente trazable Camanchaca listo.


In [6]:
# ============================================================
# SECCIÓN 5: EJECUCIÓN CON TRAZABILIDAD
# Simula consultas operativas y registra trazas completas
# ============================================================

mensajes_prueba = [
    "¿Cuál es el clima actual en Ensenada?",
    "¿Es seguro hacer cosecha en Puelche hoy?",
    "",  # mensaje vacío - debe fallar en validación
    "¿Cuáles son las condiciones actuales en Huito?",
]

print("=" * 60)
print("DEMOSTRACIÓN: Trazabilidad en el Agente Camanchaca")
print("=" * 60)

for msg in mensajes_prueba:
    traza = agente_trazable.procesar(msg)
    print(f"\n[Traza {traza.trace_id}] Entrada: {msg!r}")
    print(f"  Duración total: {traza.duracion_total_ms():.2f} ms")
    for evento in traza.eventos:
        indicador = "OK " if evento.estado == "ok" else "ERR"
        print(f"    [{indicador}] {evento.etapa}: {evento.duracion_ms:.2f} ms - {evento.detalle}")
    if msg.strip():
        time.sleep(7)  # Evita RateLimitError de GitHub Models (10 req/min)


DEMOSTRACIÓN: Trazabilidad en el Agente Camanchaca

[Traza 18c649b9-3ae] Entrada: '¿Cuál es el clima actual en Ensenada?'
  Duración total: 4202.91 ms
    [OK ] validacion_entrada: 0.00 ms - Longitud: 37 caracteres
    [OK ] identificacion_centro: 0.00 ms - Centro detectado: ensenada
    [OK ] invocacion_agente: 4202.91 ms - Modelo: gpt-4o, longitud_respuesta: 172

[Traza 2c56079b-b1b] Entrada: '¿Es seguro hacer cosecha en Puelche hoy?'
  Duración total: 3682.70 ms
    [OK ] validacion_entrada: 0.00 ms - Longitud: 40 caracteres
    [OK ] identificacion_centro: 0.00 ms - Centro detectado: puelche
    [OK ] invocacion_agente: 3682.70 ms - Modelo: gpt-4o, longitud_respuesta: 138

[Traza b3b77195-00e] Entrada: ''
  Duración total: 0.00 ms
    [ERR] validacion_entrada: 0.00 ms - Longitud: 0 caracteres

[Traza b4d97802-3b2] Entrada: '¿Cuáles son las condiciones actuales en Huito?'
  Duración total: 3581.34 ms
    [OK ] validacion_entrada: 0.00 ms - Longitud: 46 caracteres
    [OK ] identific

In [7]:
# ============================================================
# SECCIÓN 6: EJEMPLO DE TRAZA COMPLETA EN JSON
# Útil para auditoría y depuración
# ============================================================

print("=== EJEMPLO DE TRAZA COMPLETA (JSON) ===\n")
print(agente_trazable.historial_trazas[0].a_json())


=== EJEMPLO DE TRAZA COMPLETA (JSON) ===

{
  "trace_id": "18c649b9-3ae",
  "timestamp": "2026-06-15T23:18:06.549547",
  "mensaje_entrada": "¿Cuál es el clima actual en Ensenada?",
  "eventos": [
    {
      "etapa": "validacion_entrada",
      "inicio_ms": 0.0,
      "duracion_ms": 0.0,
      "estado": "ok",
      "detalle": "Longitud: 37 caracteres"
    },
    {
      "etapa": "identificacion_centro",
      "inicio_ms": 0.01,
      "duracion_ms": 0.0,
      "estado": "ok",
      "detalle": "Centro detectado: ensenada"
    },
    {
      "etapa": "invocacion_agente",
      "inicio_ms": 0.02,
      "duracion_ms": 4202.91,
      "estado": "ok",
      "detalle": "Modelo: gpt-4o, longitud_respuesta: 172"
    }
  ],
  "respuesta_final": "El clima actual en Ensenada (Piscicultura Petrohué) es el siguiente:\n\n- **Temperatura:** 7.1°C\n- **Viento:** 0.3 km/h\n- **Precipitación:** 0.0 mm\n- **Condición:** Despejado"
}


In [8]:
# ============================================================
# SECCIÓN 7: ANALIZADOR DE TRAZAS
# Identifica puntos de falla y etapas más lentas (IE10)
# ============================================================

class AnalizadorTrazasCamanchaca:
    """Analiza un conjunto de trazas del agente Camanchaca y genera un resumen."""

    @staticmethod
    def resumir(trazas: List[Traza]) -> dict:
        total   = len(trazas)
        errores = sum(1 for t in trazas if any(e.estado == "error" for e in t.eventos))
        duraciones = [t.duracion_total_ms() for t in trazas]

        tiempos_por_etapa: dict = {}
        errores_por_etapa: dict = {}
        for t in trazas:
            for e in t.eventos:
                tiempos_por_etapa.setdefault(e.etapa, []).append(e.duracion_ms)
                if e.estado == "error":
                    errores_por_etapa[e.etapa] = errores_por_etapa.get(e.etapa, 0) + 1

        promedio_por_etapa = {
            etapa: round(sum(vals) / len(vals), 2)
            for etapa, vals in tiempos_por_etapa.items()
        }

        etapa_mas_lenta = max(promedio_por_etapa, key=promedio_por_etapa.get) if promedio_por_etapa else None

        return {
            "total_trazas":          total,
            "trazas_con_error":      errores,
            "tasa_exito_pct":        round(((total - errores) / total) * 100, 1) if total else 0,
            "duracion_promedio_ms":  round(sum(duraciones) / len(duraciones), 2) if duraciones else 0,
            "duracion_maxima_ms":    round(max(duraciones), 2) if duraciones else 0,
            "promedio_por_etapa_ms": promedio_por_etapa,
            "etapa_mas_lenta":       etapa_mas_lenta,
            "errores_por_etapa":     errores_por_etapa,
        }


print("=== ANÁLISIS DE TRAZAS - AGENTE CAMANCHACA ===\n")
resumen_trazas = AnalizadorTrazasCamanchaca.resumir(agente_trazable.historial_trazas)
print(json.dumps(resumen_trazas, indent=2, ensure_ascii=False))


=== ANÁLISIS DE TRAZAS - AGENTE CAMANCHACA ===

{
  "total_trazas": 4,
  "trazas_con_error": 1,
  "tasa_exito_pct": 75.0,
  "duracion_promedio_ms": 2866.74,
  "duracion_maxima_ms": 4202.91,
  "promedio_por_etapa_ms": {
    "validacion_entrada": 0.0,
    "identificacion_centro": 0.0,
    "invocacion_agente": 3822.31
  },
  "etapa_mas_lenta": "invocacion_agente",
  "errores_por_etapa": {
    "validacion_entrada": 1
  }
}


In [9]:
# ============================================================
# SECCIÓN 8: IDENTIFICACIÓN DE PUNTOS DE FALLA Y MEJORAS
# IE10: análisis de registros para identificar fallas/mejoras
# ============================================================

print("=== PUNTOS DE FALLA IDENTIFICADOS ===\n")

if resumen_trazas["trazas_con_error"] > 0:
    print(f"⚠️  Se detectaron {resumen_trazas['trazas_con_error']} traza(s) con error.")
    for etapa, cantidad in resumen_trazas["errores_por_etapa"].items():
        print(f"   - Etapa '{etapa}': {cantidad} error(es)")
    print("\n   Causa más común: mensaje de entrada vacío (no maneja correctamente "
          "consultas sin contenido del operador).")
else:
    print("✅ No se detectaron errores en las trazas analizadas.")

print(f"\n=== ETAPA MÁS LENTA ===")
print(f"La etapa '{resumen_trazas['etapa_mas_lenta']}' presenta el mayor tiempo promedio "
      f"({resumen_trazas['promedio_por_etapa_ms'][resumen_trazas['etapa_mas_lenta']]} ms), "
      f"correspondiente a la invocación del LLM y sus herramientas — esperado, dado que "
      f"involucra llamadas a la API de GitHub Models y a Open-Meteo.")

print("\n=== OPORTUNIDADES DE MEJORA DETECTADAS ===")
print("1. Validar mensajes vacíos ANTES de invocar el agente, evitando trazas con error.")
print("2. La etapa 'invocacion_agente' concentra casi toda la latencia: se podría aplicar")
print("   cache de respuestas para consultas repetidas (ver notebook IL3.4).")
print("3. Registrar el nombre exacto de la herramienta invocada por el agente para una")
print("   trazabilidad más granular en producción.")


=== PUNTOS DE FALLA IDENTIFICADOS ===

⚠️  Se detectaron 1 traza(s) con error.
   - Etapa 'validacion_entrada': 1 error(es)

   Causa más común: mensaje de entrada vacío (no maneja correctamente consultas sin contenido del operador).

=== ETAPA MÁS LENTA ===
La etapa 'invocacion_agente' presenta el mayor tiempo promedio (3822.31 ms), correspondiente a la invocación del LLM y sus herramientas — esperado, dado que involucra llamadas a la API de GitHub Models y a Open-Meteo.

=== OPORTUNIDADES DE MEJORA DETECTADAS ===
1. Validar mensajes vacíos ANTES de invocar el agente, evitando trazas con error.
2. La etapa 'invocacion_agente' concentra casi toda la latencia: se podría aplicar
   cache de respuestas para consultas repetidas (ver notebook IL3.4).
3. Registrar el nombre exacto de la herramienta invocada por el agente para una
   trazabilidad más granular en producción.


## Conclusión - IL3.2 / IE10

Este notebook implementó **trazabilidad** completa para el agente Camanchaca:

- Cada consulta genera un **trace ID** único y una línea temporal de eventos (validación, identificación de centro, invocación del agente).
- Las trazas se almacenan en formato **JSON estructurado**, permitiendo auditoría posterior.
- El **analizador de trazas** identificó la etapa más lenta (invocación del agente/LLM) y un punto de falla (mensajes vacíos no validados antes de invocar el agente).

Estos hallazgos retroalimentan directamente las propuestas de mejora del notebook IL3.4 (Escalabilidad y Sostenibilidad) y refuerzan los protocolos de seguridad del notebook IL3.3.